<a href="https://colab.research.google.com/github/sabihadudhia/Thesis-Hallucination-Benchmarks/blob/main/HaluEval_Gemma_3_12B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
!pip install -q transformers datasets accelerate bitsandbytes sentencepiece huggingface_hub tqdm pandas numpy scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 31.4 MB/s eta 0:00:00


In [2]:
# ============================================================
# 2. IMPORTS
# ============================================================
import os, sys, json, random, subprocess, re
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoTokenizer, Gemma3ForConditionalGeneration, BitsAndBytesConfig

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA L4


In [3]:
# ============================================================
# 3. REPRODUCIBILITY
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Random seed:", SEED)

Random seed: 42


In [4]:
# ============================================================
# 4. CLONE HALUEVAL (skip if already present)
# ============================================================
if not os.path.exists("HaluEval"):
    subprocess.run(["git", "clone", "https://github.com/RUCAIBox/HaluEval.git", "HaluEval"], check=True)
halueval_commit = subprocess.check_output(["git", "-C", "HaluEval", "rev-parse", "HEAD"]).decode().strip()
print("HaluEval commit:", halueval_commit)


HaluEval commit: b7253db3cdaa0ab2c382f92b26b390109174f77e


In [5]:
# ============================================================
# 5. LOAD AND SAMPLE QA DATA
# ============================================================
with open('HaluEval/data/qa_data.json') as f:
    all_samples = [json.loads(line) for line in f]

random.seed(SEED)
sampled = random.sample(all_samples, 300)

instances = []
for s in sampled:
    instances.append({"knowledge": s["knowledge"], "question": s["question"],
                       "answer": s["right_answer"], "is_hallucinated_gt": False})
    instances.append({"knowledge": s["knowledge"], "question": s["question"],
                       "answer": s["hallucinated_answer"], "is_hallucinated_gt": True})

print(f"Total instances: {len(instances)}")
print(instances[0])

Total instances: 600
{'knowledge': ' She is best known as a featured guest vocalist on several "Billboard" Hot 100 charting songs, such as G-Eazy\'s "Me, Myself & I", David Guetta\'s "Hey Mama", Martin Garrix\'s "In the Name of Love" and Cash Cash\'s "Take Me Home"."Hey Mama" is a song by French DJ and record producer David Guetta, featuring vocals from Nicki Minaj and Bebe Rexha, as well as production from Dutch DJ and producer Afrojack.', 'question': 'Bebe Rexha was a singer who guested on the David Guetta song that was produced by which Dutch DJ?', 'answer': 'Afrojack', 'is_hallucinated_gt': False}


In [6]:
# ============================================================
# 6. HUGGING FACE LOGIN AND LOAD GEMMA-3-12B
# ============================================================
from huggingface_hub import login
login()

MODEL_ID = "google/gemma-3-12b-it"
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

print("Loading:", MODEL_ID)
model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
print("Model loaded.")

model_device = next(model.parameters()).device
print("Model device:", model_device)


Loading: google/gemma-3-12b-it


config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/109k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Model loaded.
Model device: cuda:0


In [7]:
# ============================================================
# 7. PROMPT FORMAT
# ============================================================
def build_prompt(instance):
    """Creates the context shown to the model before the Yes/No judgment."""
    return (
        f"Knowledge: {instance['knowledge']}\n"
        f"Question: {instance['question']}\n"
        f"Answer: {instance['answer']}\n"
        f"Is the answer hallucinated?\n\nJudgment:"
    )


In [8]:
# ============================================================
# 8. SCORE ONE CANDIDATE (Yes or No)
# ============================================================
@torch.no_grad()
def score_answer_choice(prompt, answer_choice):
    """
    Calculates the conditional log-probability of a candidate judgment:
    log P(answer_choice | prompt)
    """
    full_text = prompt + " " + answer_choice

    prompt_tokens = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
    full_tokens = tokenizer(full_text, return_tensors="pt", add_special_tokens=True)

    prompt_ids = prompt_tokens["input_ids"]
    full_ids = full_tokens["input_ids"]

    if full_ids.shape[1] <= prompt_ids.shape[1]:
        raise ValueError("Candidate did not add any tokens.")

    full_inputs = {key: value.to(model_device) for key, value in full_tokens.items()}
    outputs = model(**full_inputs)
    logits = outputs.logits
    log_probs = torch.log_softmax(logits, dim=-1)

    full_ids = full_ids.to(model_device)
    candidate_start = prompt_ids.shape[1]
    candidate_token_ids = full_ids[0, candidate_start:]

    if candidate_token_ids.numel() == 0:
        raise ValueError("No candidate tokens found.")

    candidate_log_probs = []
    for token_position, token_id in enumerate(candidate_token_ids, start=candidate_start):
        prediction_position = token_position - 1
        token_log_prob = log_probs[0, prediction_position, token_id]
        candidate_log_probs.append(token_log_prob)

    candidate_log_probs = torch.stack(candidate_log_probs)
    return candidate_log_probs.sum().item()


In [9]:
# ============================================================
# 9. TEST ONE INSTANCE
# ============================================================
import logging
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

test_instance = instances[0]
test_prompt = build_prompt(test_instance)

yes_score = score_answer_choice(test_prompt, "Yes")
no_score = score_answer_choice(test_prompt, "No")

print("Prompt:\n", test_prompt)
print(f"\nYes: {yes_score:.4f}")
print(f"No:  {no_score:.4f}")
predicted = yes_score > no_score
print(f"\nPredicted hallucinated: {predicted}")
print(f"Ground truth hallucinated: {test_instance['is_hallucinated_gt']}")


Prompt:
 Knowledge:  She is best known as a featured guest vocalist on several "Billboard" Hot 100 charting songs, such as G-Eazy's "Me, Myself & I", David Guetta's "Hey Mama", Martin Garrix's "In the Name of Love" and Cash Cash's "Take Me Home"."Hey Mama" is a song by French DJ and record producer David Guetta, featuring vocals from Nicki Minaj and Bebe Rexha, as well as production from Dutch DJ and producer Afrojack.
Question: Bebe Rexha was a singer who guested on the David Guetta song that was produced by which Dutch DJ?
Answer: Afrojack
Is the answer hallucinated?

Judgment:

Yes: -12.2500
No:  -0.4922

Predicted hallucinated: False
Ground truth hallucinated: False


In [10]:
# ============================================================
# 10. RUN COMPLETE HALUEVAL QA EVALUATION
# ============================================================
results = []
for instance_id, instance in enumerate(tqdm(instances, desc="Evaluating HaluEval QA")):
    prompt = build_prompt(instance)

    yes_score = score_answer_choice(prompt, "Yes")
    no_score = score_answer_choice(prompt, "No")

    predicted_hallucinated = yes_score > no_score
    correct = (predicted_hallucinated == instance["is_hallucinated_gt"])

    result = {
        "instance_id": instance_id,
        "knowledge": instance["knowledge"],
        "question": instance["question"],
        "answer": instance["answer"],
        "is_hallucinated_gt": instance["is_hallucinated_gt"],
        "yes_logprob": yes_score,
        "no_logprob": no_score,
        "predicted_hallucinated": predicted_hallucinated,
        "correct": bool(correct),
    }
    results.append(result)

Evaluating HaluEval QA:   0%|          | 0/600 [00:00<?, ?it/s]

In [11]:
# ============================================================
# 11. RESULTS DATAFRAME
# ============================================================
results_df = pd.DataFrame(results)
print("Number of evaluated instances:", len(results_df))
print("Correct:", results_df["correct"].sum())
print("Incorrect:", (~results_df["correct"]).sum())

Number of evaluated instances: 600
Correct: 393
Incorrect: 207


In [12]:
# ============================================================
# 12. ACCURACY
# ============================================================
accuracy = results_df["correct"].mean()
print(f"HaluEval QA accuracy: {accuracy:.4%}")


HaluEval QA accuracy: 65.5000%


In [13]:
# ============================================================
# 13. 95% BOOTSTRAP CONFIDENCE INTERVAL
# ============================================================
BOOTSTRAP_SEED = 42
N_BOOTSTRAPS = 10000
rng = np.random.default_rng(BOOTSTRAP_SEED)
correct_values = results_df["correct"].astype(int).to_numpy()

bootstrap_accuracies = []
for _ in range(N_BOOTSTRAPS):
    sample = rng.choice(correct_values, size=len(correct_values), replace=True)
    bootstrap_accuracies.append(sample.mean())

lower = np.percentile(bootstrap_accuracies, 2.5)
upper = np.percentile(bootstrap_accuracies, 97.5)
print(f"Accuracy: {accuracy:.4%}")
print(f"95% bootstrap CI: [{lower:.4%}, {upper:.4%}]")


Accuracy: 65.5000%
95% bootstrap CI: [61.6667%, 69.3333%]


In [14]:
# ============================================================
# 14. CONFUSION MATRIX / PER-CLASS ACCURACY
# ============================================================
# Since the 600 instances are perfectly balanced (300 right_answer, 300 hallucinated_answer),
# per-class accuracy shows whether the model is biased toward one judgment.
hallucinated_subset = results_df[results_df["is_hallucinated_gt"] == True]
not_hallucinated_subset = results_df[results_df["is_hallucinated_gt"] == False]

print(f"Accuracy on hallucinated instances (should predict 'Yes'): {hallucinated_subset['correct'].mean():.4%}")
print(f"Accuracy on non-hallucinated instances (should predict 'No'): {not_hallucinated_subset['correct'].mean():.4%}")

true_positives = ((results_df["predicted_hallucinated"] == True) & (results_df["is_hallucinated_gt"] == True)).sum()
false_positives = ((results_df["predicted_hallucinated"] == True) & (results_df["is_hallucinated_gt"] == False)).sum()
true_negatives = ((results_df["predicted_hallucinated"] == False) & (results_df["is_hallucinated_gt"] == False)).sum()
false_negatives = ((results_df["predicted_hallucinated"] == False) & (results_df["is_hallucinated_gt"] == True)).sum()

print("\nConfusion matrix:")
print(f"                 Predicted Yes   Predicted No")
print(f"Actual Yes       {true_positives:<15}{false_negatives}")
print(f"Actual No        {false_positives:<15}{true_negatives}")


Accuracy on hallucinated instances (should predict 'Yes'): 31.3333%
Accuracy on non-hallucinated instances (should predict 'No'): 99.6667%

Confusion matrix:
                 Predicted Yes   Predicted No
Actual Yes       94             206
Actual No        1              299


In [15]:
# ============================================================
# 15. INSPECT ONE PREDICTION
# ============================================================
def print_result(row):
    print("=" * 80)
    print("Instance ID:", row["instance_id"])
    print("\nKnowledge:", row["knowledge"][:200], "...")
    print("\nQuestion:", row["question"])
    print("Answer shown:", row["answer"])
    print(f"\nYes log-prob: {row['yes_logprob']:.4f}")
    print(f"No log-prob:  {row['no_logprob']:.4f}")
    print("\nGround truth hallucinated:", row["is_hallucinated_gt"])
    print("Predicted hallucinated:", row["predicted_hallucinated"])
    print("Result:", "CORRECT" if row["correct"] else "INCORRECT")

print_result(results_df.iloc[0])


Instance ID: 0

Knowledge:  She is best known as a featured guest vocalist on several "Billboard" Hot 100 charting songs, such as G-Eazy's "Me, Myself & I", David Guetta's "Hey Mama", Martin Garrix's "In the Name of Love" and C ...

Question: Bebe Rexha was a singer who guested on the David Guetta song that was produced by which Dutch DJ?
Answer shown: Afrojack

Yes log-prob: -12.2500
No log-prob:  -0.4922

Ground truth hallucinated: False
Predicted hallucinated: False
Result: CORRECT


In [16]:
# ============================================================
# 16. RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)

print(
    "Number of evaluated questions:",
    len(results_df)
)

print(
    "Correct:",
    results_df["correct"].sum()
)

print(
    "Incorrect:",
    (~results_df["correct"]).sum()
)

Number of evaluated questions: 600
Correct: 393
Incorrect: 207


In [17]:
# ============================================================
# 16. SAVE RESULTS TO DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/thesis_results', exist_ok=True)

RESULTS_FILE = "/content/drive/MyDrive/thesis_results/halueval_gemma3_12b_qa_results.jsonl"
CSV_FILE = "/content/drive/MyDrive/thesis_results/halueval_gemma3_12b_qa_results.csv"
METADATA_FILE = "/content/drive/MyDrive/thesis_results/halueval_gemma3_12b_qa_metadata.json"

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    for result in results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")
print("Saved:", RESULTS_FILE)

csv_df = results_df.copy()
csv_df.to_csv(CSV_FILE, index=False, encoding="utf-8")
print("Saved:", CSV_FILE)

Mounted at /content/drive
Saved: /content/drive/MyDrive/thesis_results/halueval_gemma3_12b_qa_results.jsonl
Saved: /content/drive/MyDrive/thesis_results/halueval_gemma3_12b_qa_results.csv


In [18]:
# ============================================================
# 17. EXPERIMENT METADATA
# ============================================================
metadata = {
    "experiment": "HaluEval QA evaluation",
    "benchmark": "HaluEval",
    "task": "QA subset, Yes/No hallucination judgment",
    "sampling": "300 knowledge/question pairs (seed=42) -> 600 paired evaluation instances (right_answer + hallucinated_answer)",
    "number_of_instances": len(instances),
    "model": MODEL_ID,
    "quantization": "8-bit",
    "seed": SEED,
    "scoring_method": "Log-probability comparison of 'Yes' vs 'No' completions; higher-scoring judgment selected.",
    "generation": False,
    "prompt_template": "Knowledge: {knowledge}\\nQuestion: {question}\\nAnswer: {answer}\\nIs the answer hallucinated?\\n\\nJudgment:",
    "software": {
        "python": sys.version,
        "pytorch": torch.__version__,
        "transformers": __import__("transformers").__version__,
    },
    "hardware": {
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    "halueval_commit": halueval_commit,
    "evaluation_timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print("Saved:", METADATA_FILE)


Saved: /content/drive/MyDrive/thesis_results/halueval_gemma3_12b_qa_metadata.json


In [19]:
# ============================================================
# 18. FINAL SUMMARY
# ============================================================
print("=" * 70)
print("HALUEVAL QA EVALUATION SUMMARY")
print("=" * 70)
print(f"Model: {MODEL_ID}")
print(f"Instances: {len(results_df)}")
print(f"Quantisation: 8-bit")
print(f"Seed: {SEED}")
print()
print(f"Accuracy: {accuracy:.4%}")
print(f"95% bootstrap CI: [{lower:.4%}, {upper:.4%}]")
print(f"Accuracy on hallucinated instances: {hallucinated_subset['correct'].mean():.4%}")
print(f"Accuracy on non-hallucinated instances: {not_hallucinated_subset['correct'].mean():.4%}")
print()
print("Files:")
print("-", RESULTS_FILE)
print("-", CSV_FILE)
print("-", METADATA_FILE)

HALUEVAL QA EVALUATION SUMMARY
Model: google/gemma-3-12b-it
Instances: 600
Quantisation: 8-bit
Seed: 42

Accuracy: 65.5000%
95% bootstrap CI: [61.6667%, 69.3333%]
Accuracy on hallucinated instances: 31.3333%
Accuracy on non-hallucinated instances: 99.6667%

Files:
- /content/drive/MyDrive/thesis_results/halueval_gemma3_12b_qa_results.jsonl
- /content/drive/MyDrive/thesis_results/halueval_gemma3_12b_qa_results.csv
- /content/drive/MyDrive/thesis_results/halueval_gemma3_12b_qa_metadata.json
